In [ ]:
# Ячейка 1: Установка библиотек и инициализация клиента (обновленная)
!pip install -q -U google-genai pandas pydantic

import pandas as pd
from google import genai
from google.genai import types
from google.colab import userdata
from pydantic import BaseModel, Field
from typing import List

# Инициализация клиента с актуальной моделью
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-3.5-flash-lite"

print("Клиент успешно инициализирован с новой моделью!")

Клиент успешно инициализирован!


In [ ]:
from google import genai
from google.colab import userdata

# Получаем ключ из секретов Colab и инициализируем клиент
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-3.5-flash-lite"

print("Клиент успешно инициализирован!")

Клиент успешно инициализирован!


In [ ]:
# Ячейка 2: Создание датасета истории просмотров (исправленная)
import random
import pandas as pd

def generate_watch_history(n=30, seed=42):
    rng = random.Random(seed)
    users = ["Алексей", "Мария", "Иван", "Ольга", "Дмитрий"]
    movies = [
        ("Интерстеллар", "Фантастика"), ("Начало", "Фантастика"),
        ("Мстители", "Боевик"), ("Матрица", "Фантастика"),
        ("Побег из Шоушенка", "Драма"), ("Зеленая миля", "Драма"),
        ("Мальчишник в Вегасе", "Комедия"), ("Один дома", "Комедия"),
        ("Заклятие", "Ужасы"), ("Сияние", "Ужасы")
    ]

    data = []
    for i in range(n):
        user = rng.choice(users)
        movie, genre = rng.choice(movies)
        rating = rng.choices([5, 4, 3, 2, 1], weights=[40, 30, 15, 10, 5])[0]
        watch_time = rng.randint(20, 150)
        completion = rng.choice([30, 50, 80, 95, 100])

        data.append({
            "user_id": user,
            "movie_title": movie,
            "genre": genre,
            "rating": rating,
            "watch_time_minutes": watch_time,
            "completion_percentage": completion
        })
    return pd.DataFrame(data)

df_history = generate_watch_history()
display(df_history.head(10))

,user_id,movie_title,genre,rating,watch_time_minutes,completion_percentage
0,Алексей,Интерстеллар,Фантастика,3,82,50
1,Мария,Начало,Фантастика,4,42,100
2,Ольга,Интерстеллар,Фантастика,5,75,50
3,Дмитрий,Сияние,Ужасы,5,70,100
4,Ольга,Матрица,Фантастика,4,91,30
5,Мария,Мальчишник в Вегасе,Комедия,5,59,50
6,Иван,Начало,Фантастика,5,44,80
7,Иван,Сияние,Ужасы,5,31,95
8,Дмитрий,Начало,Фантастика,1,116,30
9,Дмитрий,Побег из Шоушенка,Драма,3,112,100


In [ ]:
# Ячейка 3: Анализ предпочтений по пользователям
def build_user_profiles(df):
    profiles = {}
    for user, group in df.groupby("user_id"):
        # Считаем среднюю оценку и процент досмотра по жанрам
        genre_stats = group.groupby("genre").agg(
            avg_rating=("rating", "mean"),
            avg_completion=("completion_percentage", "mean"),
            total_watched=("movie_title", "count")
        ).reset_index()

        # Составим текстовый профиль
        liked = genre_stats[genre_stats["avg_completion"] >= 70]["genre"].tolist()
        watched_movies = group["movie_title"].unique().tolist()

        profiles[user] = {
            "liked_genres": liked,
            "watched_movies": watched_movies,
            "stats": genre_stats.to_dict(orient="records")
        }
    return profiles

user_profiles = build_user_profiles(df_history)
for user, profile in user_profiles.items():
    print(f"Пользователь: {user}")
    print(f"  Любимые жанры: {profile['liked_genres']}")
    print(f"  Уже смотрел: {profile['watched_movies']}\n")

Пользователь: Алексей
  Любимые жанры: []
  Уже смотрел: ['Интерстеллар', 'Матрица', 'Зеленая миля', 'Начало']

Пользователь: Дмитрий
  Любимые жанры: ['Боевик', 'Драма', 'Ужасы']
  Уже смотрел: ['Сияние', 'Начало', 'Побег из Шоушенка', 'Зеленая миля', 'Один дома', 'Мстители', 'Заклятие']

Пользователь: Иван
  Любимые жанры: ['Комедия', 'Ужасы', 'Фантастика']
  Уже смотрел: ['Начало', 'Сияние', 'Мальчишник в Вегасе']

Пользователь: Мария
  Любимые жанры: ['Боевик', 'Комедия', 'Фантастика']
  Уже смотрел: ['Начало', 'Мальчишник в Вегасе', 'Мстители', 'Зеленая миля']

Пользователь: Ольга
  Любимые жанры: ['Боевик', 'Драма']
  Уже смотрел: ['Интерстеллар', 'Матрица', 'Зеленая миля', 'Мстители', 'Начало']



In [ ]:
# Ячейка 4: Описание Pydantic-схемы
class MovieRecommendation(BaseModel):
    movie_title: str = Field(description="Название рекомендуемого фильма")
    genre: str = Field(description="Жанр фильма")
    match_score: int = Field(ge=1, le=100, description="Процент соответствия вкусам пользователя от 1 до 100")
    reasoning: str = Field(description="Обоснование рекомендации на основе истории просмотров")

class RecommendationResponse(BaseModel):
    user_id: str
    recommendations: List[MovieRecommendation] = Field(min_length=3, max_length=5, description="Список из 3-5 рекомендаций")

In [ ]:
# Ячейка 5: Функция запроса к API с гарантированным JSON
import time

def get_recommendations(user_id, profile):
    prompt = f"""
    Ты профессиональный кинокритик и рекомендательный алгоритм онлайн-кинотеатра.
    Проанализируй профиль пользователя:
    - Любимые жанры (высокий процент досмотра): {profile['liked_genres']}
    - Фильмы, которые он уже смотрел: {profile['watched_movies']}

    Задача: порекомендуй пользователю 3 новых фильма, которые он еще не смотрел, но которые точно соответствуют его вкусам.
    Заполни структуру ответа строго по схеме.
    """

    config = types.GenerateContentConfig(
        temperature=0.2,
        response_mime_type="application/json",
        response_schema=RecommendationResponse,
        max_output_tokens=1500
    )

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=config
    )

    return response.text

In [ ]:
import json
import time

results = []

for user_id, profile in user_profiles.items():
    print(f"Начинаем генерацию для {user_id}...")
    try:
        raw_json = get_recommendations(user_id, profile)
        data = json.loads(raw_json)
        for rec in data.get("recommendations", []):
            results.append({
                "user_id": user_id,
                "recommended_movie": rec["movie_title"],
                "genre": rec["genre"],
                "match_score": rec["match_score"],
                "reasoning": rec["reasoning"]
            })
    except Exception as e:
        print(f"Ошибка для {user_id}: {e}")
    time.sleep(3) # пауза между запросами, чтобы избежать ошибки 429

df_results = pd.DataFrame(results)
df_results.to_csv("recommendations_results.csv", index=False)
print("\nГотово! Рекомендации сохранены в recommendations_results.csv")
display(df_results.head(10))

Начинаем генерацию для Алексей...
Начинаем генерацию для Дмитрий...
Начинаем генерацию для Иван...
Начинаем генерацию для Мария...
Начинаем генерацию для Ольга...

Готово! Рекомендации сохранены в recommendations_results.csv


,user_id,recommended_movie,genre,match_score,reasoning
0,Алексей,Бегущий по лезвию 2049,Фантастика,95,Пользователь высоко оценил глубокие научно-фан...
1,Алексей,Побег из Шоушенка,Драма,92,Учитывая положительную историю просмотра фильм...
2,Алексей,Тёмный рыцарь,Боевик,88,Фильм снят тем же режиссером Кристофером Нолан...
3,Дмитрий,Темный рыцарь,Боевик,98,Пользователь высоко оценил сложные сюжеты в фи...
4,Дмитрий,Психо,Ужасы,95,Учитывая любовь пользователя к классическим и ...
5,Дмитрий,Список Шиндлера,Драма,96,Пользователь предпочитает глубокие и эмоционал...
6,Иван,Интерстеллар,Фантастика,95,Учитывая вашу историю просмотров фильма 'Начал...
7,Иван,Мальчишник 2: Из Вегаса в Бангкок,Комедия,90,Поскольку вам понравился 'Мальчишник в Вегасе'...
8,Иван,Нечто,Ужасы,88,"Этот фильм сочетает в себе элементы ужасов, ко..."
9,Мария,Стражи Галактики,Фантастика,95,Этот фильм идеально сочетает в себе динамичный...


In [ ]:
import random
import pandas as pd
from google.genai import types
from pydantic import BaseModel, Field
from typing import List
import json
import time

# 1. Генерируем историю просмотров
def generate_watch_history(n=30, seed=42):
    rng = random.Random(seed)
    users = ["Алексей", "Мария", "Иван", "Ольга", "Дмитрий"]
    movies = [
        ("Интерстеллар", "Фантастика"), ("Начало", "Фантастика"),
        ("Мстители", "Боевик"), ("Матрица", "Фантастика"),
        ("Побег из Шоушенка", "Драма"), ("Зеленая миля", "Драма"),
        ("Мальчишник в Вегасе", "Комедия"), ("Один дома", "Комедия"),
        ("Заклятие", "Ужасы"), ("Сияние", "Ужасы")
    ]
    data = []
    for i in range(n):
        user = rng.choice(users)
        movie, genre = rng.choice(movies)
        rating = rng.choices([5, 4, 3, 2, 1], weights=[40, 30, 15, 10, 5])[0]
        watch_time = rng.randint(20, 150)
        completion = rng.choice([30, 50, 80, 95, 100])
        data.append({
            "user_id": user, "movie_title": movie, "genre": genre,
            "rating": rating, "watch_time_minutes": watch_time,
            "completion_percentage": completion
        })
    return pd.DataFrame(data)

df_history = generate_watch_history()

# 2. Строим профили пользователей
def build_user_profiles(df):
    profiles = {}
    for user, group in df.groupby("user_id"):
        genre_stats = group.groupby("genre").agg(
            avg_rating=("rating", "mean"),
            avg_completion=("completion_percentage", "mean"),
            total_watched=("movie_title", "count")
        ).reset_index()
        liked = genre_stats[genre_stats["avg_completion"] >= 70]["genre"].tolist()
        watched_movies = group["movie_title"].unique().tolist()
        profiles[user] = {
            "liked_genres": liked,
            "watched_movies": watched_movies,
            "stats": genre_stats.to_dict(orient="records")
        }
    return profiles

user_profiles = build_user_profiles(df_history)

# 3. Описываем Pydantic-схему
class MovieRecommendation(BaseModel):
    movie_title: str = Field(description="Название рекомендуемого фильма")
    genre: str = Field(description="Жанр фильма")
    match_score: int = Field(ge=1, le=100, description="Процент соответствия вкусам пользователя от 1 до 100")
    reasoning: str = Field(description="Обоснование рекомендации на основе истории просмотров")

class RecommendationResponse(BaseModel):
    user_id: str
    recommendations: List[MovieRecommendation] = Field(min_length=3, max_length=5, description="Список из 3-5 рекомендаций")

# 4. Функция запроса к API
def get_recommendations(user_id, profile):
    prompt = f"""
    Ты профессиональный кинокритик и рекомендательный алгоритм онлайн-кинотеатра.
    Проанализируй профиль пользователя:
    - Любимые жанры (высокий процент досмотра): {profile['liked_genres']}
    - Фильмы, которые он уже смотрел: {profile['watched_movies']}

    Задача: порекомендуй пользователю 3 новых фильма, которые он еще не смотрел, но которые точно соответствуют его вкусам.
    Заполни структуру ответа строго по схеме.
    """
    config = types.GenerateContentConfig(
        temperature=0.2,
        response_mime_type="application/json",
        response_schema=RecommendationResponse,
        max_output_tokens=1500
    )
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=config
    )
    return response.text

print("Все вспомогательные переменные и функции успешно созданы!")

Все вспомогательные переменные и функции успешно созданы!
